In [73]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import glob

LOG_DIR = "../logs/mainRun2Clients"

In [74]:


aggregation_data = pd.read_csv(f"{LOG_DIR}/he-aggregation-service-time.csv")

clients_data = pd.DataFrame()
for file_path in glob.glob(f"{LOG_DIR}/he-client-*-go-time.csv"):
    client = pd.read_csv(file_path)
    client["ServiceName"] = file_path.split("/")[-1].split("-time.")[0]
    clients_data = pd.concat([clients_data, client], ignore_index=True)

# First, parse the Runtime column to numeric values (assuming it's in string format)
def parse_duration_to_ms(time_str):
    """Parse milliseconds, microseconds, nanoseconds or seconds from a string."""
    if "ms" in time_str:
        return float(time_str.replace("ms", "").strip())
    elif "us" in time_str:
        return float(time_str.replace("us", "").strip()) / 1000.0
    elif "µs" in time_str:
        return float(time_str.replace("µs", "").strip()) / 1000.0
    elif "ns" in time_str:
        return float(time_str.replace("ns", "").strip()) / 1_000_000.0
    elif "s" in time_str:
        return float(time_str.replace("s", "").strip()) * 1000.0
    else:
        return float(time_str)

# Convert runtime to numeric
clients_data["runtime_ms"] = clients_data["Runtime"].apply(parse_duration_to_ms)
aggregation_data["runtime_ms"] = aggregation_data["Runtime"].apply(parse_duration_to_ms)

clients_data.head()

,Date,EndTime,Name,Runtime,ServiceName,runtime_ms
0,2025/07/22,16:42:53.857312,DataSpaceClientService - RegisterClient,12.761208ms,he-client-2-go,12.761208
1,2025/07/22,16:42:53.865943,HEService - SetParameters,8.578583ms,he-client-2-go,8.578583
2,2025/07/22,16:42:58.855781,HEService - PartialShareAggregation,2.333µs,he-client-2-go,0.002333
3,2025/07/22,16:42:58.863080,EncryptionHandler - handleReceivePublicKey,1.570916ms,he-client-2-go,1.570916
4,2025/07/22,16:42:59.063884,HEService - PartialRelinKeyAggregation,2.646667ms,he-client-2-go,2.646667


In [75]:
unique_time_tracks = clients_data["Name"].unique()
print(unique_time_tracks)

['DataSpaceClientService - RegisterClient' 'HEService - SetParameters'
 'HEService - PartialShareAggregation'
 'EncryptionHandler - handleReceivePublicKey'
 'HEService - PartialRelinKeyAggregation' 'HEService - Encrypt'
 'DataSpaceClientService - UploadData'
 'HEService - PublicKeySwitchGeneration'
 'EncryptionHandler - handlePublicKeySwitch'
 'EncryptionHandler - handlePublicKeySwitchAggregate']


In [76]:
clients_mean_by_name = clients_data.groupby("Name")["runtime_ms"].agg(['sum', 'mean', 'count']).reset_index()
clients_mean_by_name.columns = ['Name', 'Total_Runtime_ms', 'Mean_Runtime_ms', 'Count']
clients_mean_by_name = clients_mean_by_name.round(2)
clients_mean_by_name

,Name,Total_Runtime_ms,Mean_Runtime_ms,Count
0,DataSpaceClientService - RegisterClient,30.05,15.02,2
1,DataSpaceClientService - UploadData,7020.17,1170.03,6
2,EncryptionHandler - handlePublicKeySwitch,752.00,125.33,6
3,EncryptionHandler - handlePublicKeySwitchAggre...,0.07,0.01,6
4,EncryptionHandler - handleReceivePublicKey,4.46,1.49,3
5,HEService - Encrypt,5167.44,861.24,6
6,HEService - PartialRelinKeyAggregation,7.55,2.52,3
7,HEService - PartialShareAggregation,0.01,0.00,3
8,HEService - PublicKeySwitchGeneration,177.16,29.53,6
9,HEService - SetParameters,16.94,8.47,2


In [77]:
aggregation_mean_by_name = aggregation_data.groupby("Name")["runtime_ms"].agg(['sum', 'mean', 'count']).reset_index()
aggregation_mean_by_name.columns = ['Name', 'Total_Runtime_ms', 'Mean_Runtime_ms', 'Count']
aggregation_mean_by_name = aggregation_mean_by_name.round(2)
aggregation_mean_by_name

,Name,Total_Runtime_ms,Mean_Runtime_ms,Count
0,addClient,0.00,0.00,2
1,aggregateWeights,166.27,55.42,3
2,initiateKeySwitchGeneration,1100.46,366.82,3
3,requestClientTraining,33.12,11.04,3
4,startEncryptionSetupPhaseFor,10598.38,5299.19,2
5,updateClientModels,6648.30,2216.10,3


# Averaging the used memory

In [78]:
aggregation_memory_data = pd.read_csv(f"{LOG_DIR}/he-aggregation-service-data.csv")

clients_memory_data = pd.DataFrame()
for file_path in glob.glob(f"{LOG_DIR}/he-client-*-go-data.csv"):
    client = pd.read_csv(file_path)
    client["ServiceName"] = file_path.split("/")[-1].split("-data.")[0]
    clients_memory_data = pd.concat([clients_memory_data, client], ignore_index=True)


clients_memory_data.head()

,Date,EndTime,Name,Alloc,TotalAlloc,Sys,ServiceName
0,2025/07/22,16:42:53.857357,HEService - SetParameters,9,13,21,he-client-2-go
1,2025/07/22,16:42:53.865917,HEService - SetParameters - after public key g...,44,69,65,he-client-2-go
2,2025/07/22,16:42:58.855683,EncryptionHandler - handleSharedPublicKey,44,69,65,he-client-2-go
3,2025/07/22,16:42:58.855775,HEService - PartialShareAggregation,44,69,65,he-client-2-go
4,2025/07/22,16:42:58.855870,EncryptionHandler - handleSharedPublicKey - af...,44,69,65,he-client-2-go


In [79]:
clients_mem_mean_by_name = clients_memory_data.groupby("Name")["Alloc"].agg(['sum', 'mean', 'count']).reset_index()
clients_mem_mean_by_name.columns = ['Name', 'Total_Alloc', 'Mean_Alloc', 'Count']
clients_mem_mean_by_name = clients_mem_mean_by_name.round(2)
clients_mem_mean_by_name

,Name,Total_Alloc,Mean_Alloc,Count
0,EncryptionHandler - handlePublicKeySwitch,4354,725.67,6
1,EncryptionHandler - handlePublicKeySwitch - af...,5592,932.00,6
2,EncryptionHandler - handlePublicKeySwitchAggre...,5593,932.17,6
3,EncryptionHandler - handleReceivePublicKey,202,67.33,3
4,EncryptionHandler - handleReceivePublicKey - a...,215,71.67,3
5,EncryptionHandler - handleSharedPublicKey,192,64.00,3
6,EncryptionHandler - handleSharedPublicKey - af...,202,67.33,3
7,EncryptionHandler - handleSharedRelinKey,215,71.67,3
8,EncryptionHandler - handleSharedRelinKey - aft...,324,108.00,3
9,HEService - Encrypt,1620,270.00,6


In [80]:
aggregation_mem_mean_by_name = aggregation_memory_data.groupby("Name")["Alloc"].agg(['sum', 'mean', 'count']).reset_index()
aggregation_mem_mean_by_name.columns = ['Name', 'Total_Alloc', 'Mean_Alloc', 'Count']
aggregation_mem_mean_by_name = aggregation_mem_mean_by_name.round(2)
aggregation_mem_mean_by_name

,Name,Total_Alloc,Mean_Alloc,Count
0,ClientManagementService - EncryptionSetupBegin,86,43.00,2
1,ClientManagementService - EncryptionSetupPubli...,109,54.50,2
2,ClientManagementService - EncryptionSetupPubli...,129,64.50,2
3,ClientManagementService - EncryptionSetupRelin...,188,94.00,2
4,ClientManagmentHandler - handlePostClientData,8859,738.25,12
5,ClientManagmentHandler - handleRegisterClient,170,42.50,4
6,UpdateClients - AfterAggregation,3237,1079.00,3
7,UpdateClients - AfterKeySwitchGeneration,3675,1225.00,3
8,UpdateClients - AfterPublicKeySwitch,1913,637.67,3
9,UpdateClients - AfterPublicKeySwitchShareCalcu...,3695,1231.67,3


In [81]:
aggregation_mem_mean_by_name.to_csv(f"{LOG_DIR}/eval_aggregation_memory_mean_by_name.csv", index=False)
clients_mem_mean_by_name.to_csv(f"{LOG_DIR}/eval_clients_memory_mean_by_name.csv", index=False)
aggregation_mean_by_name.to_csv(f"{LOG_DIR}/eval_aggregation_time_mean_by_name.csv", index=False)
clients_mean_by_name.to_csv(f"{LOG_DIR}/eval_clients_time_mean_by_name.csv", index=False)